# Lesson 06: Transfer Learning with PyTorch

This notebook covers:
- What is transfer learning and why use it?
- Using pretrained models from torchvision
- Feature extraction vs fine-tuning
- Freezing and unfreezing layers
- Modifying classifier heads for custom datasets
- Training with EfficientNet-B0
- Making predictions on custom images

## What is Transfer Learning?

**Transfer learning** = using knowledge from one task to help with another task

In deep learning:
- Take a model pretrained on a large dataset (e.g., ImageNet: 1.2M images, 1000 classes)
- Adapt it to your smaller dataset (e.g., pizza/steak/sushi: 750 images, 3 classes)

## Why Transfer Learning?

1. **Less data required**: Pretrained models already learned useful features
2. **Faster training**: Start from good weights instead of random
3. **Better performance**: Benefits from large-scale pretraining
4. **Less compute**: Don't need to train from scratch

## Two Approaches:

### 1. Feature Extraction (Freezing)
- Freeze all pretrained layers (don't update weights)
- Only train new classifier head
- Fast, works well when data is similar to pretraining data

### 2. Fine-Tuning
- Unfreeze some/all layers
- Train entire model or parts of it
- Slower, but can adapt better to very different data

## Setup and Imports

In [ ]:
# Import PyTorch and torchvision
import torch
import torchvision

# Check versions
# torchvision >= 0.13 has improved pretrained model API
print(torch.__version__)
print(torchvision.__version__)

In [ ]:
# Continue with regular imports
import matplotlib.pyplot as plt
from torch import nn
from torchvision import transforms

In [ ]:
# Try to get torchinfo, install it if it doesn't work
# torchinfo provides detailed model summaries
try:
    from torchinfo import summary
except:
    print("[INFO] Couldn't find torchinfo... installing it.")
    !pip install -q torchinfo
    from torchinfo import summary

In [ ]:
# Try to import the going_modular directory
# These are our custom data and training modules from Lesson 05
try:
    from going_modular import data_setup, engine
except:
    # Get the going_modular scripts from GitHub
    print("[INFO] Couldn't find going_modular scripts... downloading them from GitHub.")
    !git clone https://github.com/mrdbourke/pytorch-deep-learning
    !mv pytorch-deep-learning/going_modular .
    !rm -rf pytorch-deep-learning
    from going_modular import data_setup, engine

In [ ]:
# Setup device agnostic code
# Use GPU if available for faster training
device = "cuda" if torch.cuda.is_available() else "cpu"
device

## Part 1: Get Data

In [ ]:
import os
import zipfile
from pathlib import Path
import requests

# Setup path to data folder
data_path = Path("data/")
image_path = data_path / "pizza_steak_sushi"

# If the image folder doesn't exist, download it and prepare it
if image_path.is_dir():
    print(f"{image_path} directory exists.")
else:
    print(f"Did not find {image_path} directory, creating one...")
    image_path.mkdir(parents=True, exist_ok=True)
    
    # Download pizza, steak, sushi data
    with open(data_path / "pizza_steak_sushi.zip", "wb") as f:
        request = requests.get("https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip")
        print("Downloading pizza, steak, sushi data...")
        f.write(request.content)

    # Unzip pizza, steak, sushi data
    with zipfile.ZipFile(data_path / "pizza_steak_sushi.zip", "r") as zip_ref:
        print("Unzipping pizza, steak, sushi data...") 
        zip_ref.extractall(image_path)

In [ ]:
# Setup directories
train_dir = image_path / "train"
test_dir = image_path / "test"

## Part 2: Create DataLoaders

### Method 1: Manual Transforms (for older torchvision versions)

In [ ]:
# Create a transforms pipeline manually (required for torchvision < 0.13)
# These are the STANDARD ImageNet preprocessing steps
manual_transforms = transforms.Compose([
    # 1. Reshape all images to 224x224
    # Most pretrained models expect this size (though some may differ)
    transforms.Resize((224, 224)),
    
    # 2. Convert images to tensors and scale to [0, 1]
    transforms.ToTensor(),
    
    # 3. Normalize with ImageNet mean and std
    # These are the statistics from ImageNet dataset
    # Mean: [0.485, 0.456, 0.406] (R, G, B channels)
    # Std: [0.229, 0.224, 0.225] (R, G, B channels)
    # IMPORTANT: Use the same normalization as the pretrained model!
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
# Create training and testing DataLoaders
# Using manually defined transforms
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=manual_transforms,  # Apply ImageNet preprocessing
    batch_size=32
)

## Part 3: Getting Pretrained Weights (torchvision >= 0.13)

Modern way to get pretrained models and their transforms.

In [ ]:
# Get a set of pretrained model weights
# .DEFAULT = best available weights from pretraining on ImageNet
# EfficientNet-B0: efficient and accurate model (good starting point)
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
weights

In [ ]:
# Get the transforms used to create our pretrained weights
# This automatically gives us the correct preprocessing pipeline!
# Much better than manually defining transforms
auto_transforms = weights.transforms()
auto_transforms

In [ ]:
# Create training and testing DataLoaders with automatic transforms
# IMPORTANT: Use the same transforms as the pretrained model
# Otherwise the model will see differently preprocessed data than it was trained on!
train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(
    train_dir=train_dir,
    test_dir=test_dir,
    transform=auto_transforms,  # Use pretrained model's transforms
    batch_size=32
)

## Part 4: Load Pretrained Model

In [ ]:
# OLD METHOD (before torchvision v0.13):
# model = torchvision.models.efficientnet_b0(pretrained=True).to(device)

# NEW METHOD (torchvision v0.13+):
# Setup the model with pretrained weights and send it to the target device
weights = torchvision.models.EfficientNet_B0_Weights.DEFAULT
model = torchvision.models.efficientnet_b0(weights=weights).to(device)

# What's happening here:
# 1. Load EfficientNet-B0 architecture
# 2. Load pretrained weights from ImageNet
# 3. Move model to GPU/CPU

## Part 5: Understanding the Model Architecture

In [ ]:
# Print a summary using torchinfo
# This shows us the model's structure, parameter counts, and trainability
summary(
    model=model,
    input_size=(1, 3, 224, 224),  # (batch_size, channels, height, width)
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

## Part 6: Freezing the Base Model (Feature Extraction)

We'll freeze the pretrained feature extractor and only train the classifier.

In [ ]:
# View the features section of the model
# This is the convolutional base that extracts features
# model.features

In [ ]:
# Freeze all parameters in the features section
# This means these layers won't be updated during training
# We're using the pretrained feature extraction as-is
for param in model.features.parameters():
    param.requires_grad = False  # Don't compute gradients for these parameters

In [ ]:
# View the classifier section
# This is what we'll modify and train
model.classifier

## Part 7: Modifying the Classifier for Our Dataset

In [ ]:
# Set the manual seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Get the number of output classes
# ImageNet has 1000 classes, we have 3 (pizza, steak, sushi)
output_shape = len(class_names)
print(output_shape)

# Recreate the classifier layer
# Original classifier outputs 1000 classes (ImageNet)
# We need to output 3 classes (our dataset)
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(p=0.2, inplace=True),  # Dropout for regularization
    # Linear layer: 1280 features (from EfficientNet) -> 3 classes (our data)
    torch.nn.Linear(in_features=1280, out_features=output_shape, bias=True)
).to(device)

In [ ]:
# Do a summary AFTER freezing features and changing the classifier
# Notice: most parameters are non-trainable (frozen)
# Only the classifier parameters are trainable
summary(
    model,
    input_size=(32, 3, 224, 224),  # Use batch_size=32 to match our DataLoader
    verbose=0,
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    row_settings=["var_names"]
)

## Part 8: Training the Model

In [ ]:
# Define loss and optimizer
# CrossEntropyLoss for multi-class classification
loss_fn = nn.CrossEntropyLoss()

# Adam optimizer
# Note: optimizer only updates parameters where requires_grad=True
# So it will only update the classifier, not the frozen features
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [ ]:
# Import our training engine from going_modular
from going_modular import engine

# Set seeds for reproducibility
torch.manual_seed(42)
torch.cuda.manual_seed(42)

# Start the timer
from timeit import default_timer as timer
start_time = timer()

# Setup training and save the results
# Training should be relatively fast since we're only updating the classifier
results = engine.train(
    model=model,
    train_dataloader=train_dataloader,
    test_dataloader=test_dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    epochs=10,
    device=device
)

# End the timer and print out how long it took
end_time = timer()
print(f"[INFO] Total training time: {end_time-start_time:.3f} seconds")

## Part 9: Visualizing Training Results

In [ ]:
# Get the plot_loss_curves() function
# This helper function plots training and test loss/accuracy curves
try:
    from helper_functions import plot_loss_curves
except:
    print("[INFO] Couldn't find helper_functions.py, downloading...")
    with open("helper_functions.py", "wb") as f:
        import requests
        request = requests.get("https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/helper_functions.py")
        f.write(request.content)
    from helper_functions import plot_loss_curves

In [ ]:
# Plot the loss curves of our model
# Should see decreasing loss and increasing accuracy
# Transfer learning often achieves good results quickly!
plot_loss_curves(results)
plt.ylim(0, 1)
plt.show()

## Part 10: Making Predictions on Custom Images

In [ ]:
from typing import List, Tuple
from PIL import Image

In [ ]:
# Create a function to predict and plot images
def pred_and_plot_image(
    model: torch.nn.Module,
    image_path: str,
    class_names: List[str],
    image_size: Tuple[int, int] = (224, 224),
    transform: torchvision.transforms = None,
    device: torch.device = device
):
    """
    Predicts on a target image with a target model.

    Args:
        model: A trained PyTorch model
        image_path: Filepath to target image
        class_names: List of class names
        image_size: Size to resize image to
        transform: Transform pipeline to apply
        device: Device to perform computation on
    """
    
    # Open image
    img = Image.open(image_path)

    # Create transformation if one doesn't exist
    if transform is not None:
        image_transform = transform
    else:
        # Use same transforms as the model was trained with
        image_transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]
            ),
        ])

    # Put model in eval mode
    model.eval()
    with torch.inference_mode():
        # Transform image and add batch dimension
        # Shape: (3, 224, 224) -> (1, 3, 224, 224)
        transformed_image = image_transform(img).unsqueeze(dim=0)
        
        # Make prediction
        target_image_pred = model(transformed_image.to(device))

    # Convert logits to prediction probabilities
    target_image_pred_probs = torch.softmax(target_image_pred, dim=1)
    
    # Convert prediction probabilities to labels
    target_image_pred_label = torch.argmax(target_image_pred_probs, dim=1)
    
    # Plot image with predicted label and probability
    plt.figure()
    plt.imshow(img)
    plt.title(
        f"Pred: {class_names[target_image_pred_label]} | "
        f"Prob: {target_image_pred_probs.max():.3f}"
    )
    plt.axis(False)

In [ ]:
# Get a random list of image paths from test set
import random

num_images_to_plot = 3

# Get all test image paths
test_image_path_list = list(Path(test_dir).glob("*/*.jpg"))

# Randomly select images to predict on
test_image_path_sample = random.sample(
    population=test_image_path_list,
    k=num_images_to_plot
)

# Make predictions on and plot the images
for image_path in test_image_path_sample:
    pred_and_plot_image(
        model=model,
        image_path=image_path,
        class_names=class_names,
        image_size=(224, 224),
        transform=auto_transforms
    )

## Part 11: Predicting on a Custom Image from the Internet

In [ ]:
# Download custom image
import requests

# Setup custom image path
custom_image_path = data_path / "04-pizza-dad.jpeg"

# Download the image if it doesn't already exist
if not custom_image_path.is_file():
    with open(custom_image_path, "wb") as f:
        # When downloading from GitHub, need to use the "raw" file link
        request = requests.get(
            "https://raw.githubusercontent.com/mrdbourke/pytorch-deep-learning/main/images/04-pizza-dad.jpeg"
        )
        print(f"Downloading {custom_image_path}...")
        f.write(request.content)
else:
    print(f"{custom_image_path} already exists, skipping download.")

In [ ]:
# Predict on custom image
pred_and_plot_image(
    model=model,
    image_path=custom_image_path,
    class_names=class_names,
    image_size=(224, 224),
    transform=auto_transforms
)